In [1]:
from langchain_core.documents import Document

c:\Coding\AI\GEN AI\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mamma-pets-docs", "id": 1}
    ),
    Document(
        page_content="Cats are independent pets, often valued for their calm and quiet nature.",
        metadata={"source": "mamma-pets-docs", "id": 2}
    ),
    Document(
        page_content="Birds are intelligent animals that can mimic sounds and require social interaction.",
        metadata={"source": "mamma-pets-docs", "id": 3}
    ),
    Document(
        page_content="Fish are low-maintenance pets that bring tranquility to home environments.",
        metadata={"source": "mamma-pets-docs", "id": 4}
    )
]

In [3]:
documents

[Document(metadata={'source': 'mamma-pets-docs', 'id': 1}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mamma-pets-docs', 'id': 2}, page_content='Cats are independent pets, often valued for their calm and quiet nature.'),
 Document(metadata={'source': 'mamma-pets-docs', 'id': 3}, page_content='Birds are intelligent animals that can mimic sounds and require social interaction.'),
 Document(metadata={'source': 'mamma-pets-docs', 'id': 4}, page_content='Fish are low-maintenance pets that bring tranquility to home environments.')]

In [28]:
## VectorStore
from langchain_chroma import Chroma
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key=groq_api_key, model ="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021D55E04FC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021D55E05940>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [22]:
import torch
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15853.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
vectorstore = Chroma.from_documents(documents,embedding=embeddings)

In [9]:
vectorstore.similarity_search("cat")

[Document(id='bd03e3c8-81a3-46e4-b411-7e8f0fa3227e', metadata={'id': 2, 'source': 'mamma-pets-docs'}, page_content='Cats are independent pets, often valued for their calm and quiet nature.'),
 Document(id='aca34fc7-034e-49e8-83dd-3a99bc1b0495', metadata={'id': 4, 'source': 'mamma-pets-docs'}, page_content='Fish are low-maintenance pets that bring tranquility to home environments.'),
 Document(id='8c8cef8e-7afc-4b5f-96dc-4fea9fb20955', metadata={'id': 1, 'source': 'mamma-pets-docs'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='0ea746dc-55fd-46e3-bbc6-e7c9a501adf1', metadata={'id': 3, 'source': 'mamma-pets-docs'}, page_content='Birds are intelligent animals that can mimic sounds and require social interaction.')]

In [10]:
## Async query 

await vectorstore.asimilarity_search("cat")

[Document(id='bd03e3c8-81a3-46e4-b411-7e8f0fa3227e', metadata={'source': 'mamma-pets-docs', 'id': 2}, page_content='Cats are independent pets, often valued for their calm and quiet nature.'),
 Document(id='aca34fc7-034e-49e8-83dd-3a99bc1b0495', metadata={'source': 'mamma-pets-docs', 'id': 4}, page_content='Fish are low-maintenance pets that bring tranquility to home environments.'),
 Document(id='8c8cef8e-7afc-4b5f-96dc-4fea9fb20955', metadata={'source': 'mamma-pets-docs', 'id': 1}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='0ea746dc-55fd-46e3-bbc6-e7c9a501adf1', metadata={'source': 'mamma-pets-docs', 'id': 3}, page_content='Birds are intelligent animals that can mimic sounds and require social interaction.')]

In [12]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='bd03e3c8-81a3-46e4-b411-7e8f0fa3227e', metadata={'source': 'mamma-pets-docs', 'id': 2}, page_content='Cats are independent pets, often valued for their calm and quiet nature.'),
  0.9816058874130249),
 (Document(id='aca34fc7-034e-49e8-83dd-3a99bc1b0495', metadata={'id': 4, 'source': 'mamma-pets-docs'}, page_content='Fish are low-maintenance pets that bring tranquility to home environments.'),
  1.434624195098877),
 (Document(id='8c8cef8e-7afc-4b5f-96dc-4fea9fb20955', metadata={'id': 1, 'source': 'mamma-pets-docs'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740900039672852),
 (Document(id='0ea746dc-55fd-46e3-bbc6-e7c9a501adf1', metadata={'id': 3, 'source': 'mamma-pets-docs'}, page_content='Birds are intelligent animals that can mimic sounds and require social interaction.'),
  1.6121470928192139)]

## Retriever

In [13]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

In [14]:
retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat", "dog"])


[[Document(id='bd03e3c8-81a3-46e4-b411-7e8f0fa3227e', metadata={'source': 'mamma-pets-docs', 'id': 2}, page_content='Cats are independent pets, often valued for their calm and quiet nature.')],
 [Document(id='8c8cef8e-7afc-4b5f-96dc-4fea9fb20955', metadata={'id': 1, 'source': 'mamma-pets-docs'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [15]:
retriever=vectorstore.as_retriever(\
    search_type = "similarity",
    search_kwargs={"k":1}
    )

retriever.batch(["cat", "dog"])


[[Document(id='bd03e3c8-81a3-46e4-b411-7e8f0fa3227e', metadata={'id': 2, 'source': 'mamma-pets-docs'}, page_content='Cats are independent pets, often valued for their calm and quiet nature.')],
 [Document(id='8c8cef8e-7afc-4b5f-96dc-4fea9fb20955', metadata={'id': 1, 'source': 'mamma-pets-docs'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only

{question}

Context:
{context}

"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

In [ ]:
rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm |

In [31]:
rag_chain.invoke("Tell me about Dogs").content

'Dogs are great companions, known for their loyalty and friendliness.'